In [ ]:
!pip install mtcnn tensorflow opencv-python keras-facenet

In [ ]:
import cv2
import numpy as np
from mtcnn import MTCNN
from keras_facenet import FaceNet
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from google.colab import drive
import os

# Mount Google Drive to access dataset
drive.mount('/content/drive')

In [ ]:
!pip install facenet-pytorch opencv-python torch torchvision

In [ ]:
import cv2
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import numpy as np
from torch.utils.data import Dataset, DataLoader
import os

In [ ]:
# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# MTCNN for detection
mtcnn = MTCNN(keep_all=True, device=device)

# FaceNet for embeddings
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

In [ ]:
from facenet_pytorch import MTCNN
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(keep_all=True, device=device)

def detect_face(frame):
    boxes, probs, landmarks = mtcnn.detect(frame, landmarks=True)
    if boxes is not None:
        return boxes[0], landmarks[0]  # Assume single face per frame
    return None, None

In [ ]:
from facenet_pytorch import InceptionResnetV1
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def get_embedding(image):
    img = torch.tensor(image).permute(2, 0, 1).unsqueeze(0).float().to(device)
    embedding = resnet(img).detach().cpu().numpy()
    return embedding[0]  # 128-D vector

In [ ]:
import cv2
import numpy as np

def align_face(image, landmarks):
    # Simple alignment using eye landmarks from MTCNN
    left_eye = landmarks[0]  # [x, y]
    right_eye = landmarks[1]
    dY = right_eye[1] - left_eye[1]
    dX = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dY, dX))
    eyes_center = ((left_eye[0] + right_eye[0]) // 2, (left_eye[1] + right_eye[1]) // 2)
    M = cv2.getRotationMatrix2D(eyes_center, angle, 1.0)
    aligned = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))
    return aligned

In [ ]:
import torch.nn as nn

class FaceClassifier(nn.Module):
    def __init__(self, input_dim=512, num_classes=5):  # 3 people + imposter
        super(FaceClassifier, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        return self.fc(x)

classifier = FaceClassifier().to(device)

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.classes = ['abhinav', 'andy', 'adarsh', 'adam', 'unknown_subjects']
        self.data = []
        for idx, cls in enumerate(self.classes):
            cls_dir = os.path.join(root_dir, cls)
            for img_name in os.listdir(cls_dir):
                self.data.append((os.path.join(cls_dir, img_name), idx))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        # print(img_path)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        boxes, _, landmarks = mtcnn.detect(img, landmarks=True)

        # Handle cases where face detection fails
        if boxes is None or landmarks is None: #Check if landmarks is also None.
            # Return a dummy tensor and label or skip the image
            # Here, we'll skip the image by returning None, None
            # and handling this in the training loop.
            # return None, None  # Original line causing the error
            # Instead of returning None, return a dummy tensor and label
            dummy_tensor = torch.zeros(3, 160, 160)  # Adjust size if needed
            return dummy_tensor, -1  # -1 can be used as a label for invalid data

            # Alternative: return a dummy tensor and label
            # dummy_tensor = torch.zeros(3, 160, 160) # Adjust size if needed
            # return dummy_tensor, -1 # -1 can be used as a label for invalid data

        x, y, w, h = boxes[0]
        # Check if the bounding box coordinates are valid before extraction
        if w > 0 and h > 0 and int(y) < img.shape[0] and int(x) < img.shape[1]:
            face = img[int(y):int(h), int(x):int(w)]
            if face.size != 0:  # Check if face is not empty
                face = cv2.resize(face, (160, 160))
                face = align_face(face, landmarks[0])  # Add alignment function here
                face = torch.tensor(face).permute(2, 0, 1).float() / 255.0
                return face, label
            else:
                # return None, None  # or return dummy data # Original line causing the error
                dummy_tensor = torch.zeros(3, 160, 160)  # Adjust size if needed
                return dummy_tensor, -1  # -1 can be used as a label for invalid data
        else:
            # return None, None  # or return dummy data # Original line causing the error
            dummy_tensor = torch.zeros(3, 160, 160)  # Adjust size if needed
            return dummy_tensor, -1  # -1 can be used as a label for invalid data

In [ ]:
# Load dataset (upload to Colab first)
dataset = FaceDataset('/content/drive/MyDrive/Face_Recog/march/enrolled_subjects')
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Training loop
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    for batch in dataloader:
        images, labels = batch
        # Skip batches containing dummy data
        if (labels == -1).any():  # Check if any label in the batch is -1
            continue
        if images is None:
            continue
        images, labels = images.to(device), labels.to(device)
        embeddings = resnet(images)
        outputs = classifier(embeddings)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

# Inference on video
cap = cv2.VideoCapture('/content/drive/MyDrive/Face_Recog/test_video/adarsh_video.mp4')
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    boxes, _, landmarks = mtcnn.detect(frame_rgb, landmarks=True)
    if boxes is not None:
        x, y, w, h = boxes[0]
        face = frame_rgb[int(y):int(h), int(x):int(w)]
        face = cv2.resize(face, (160, 160))
        face = align_face(face, landmarks[0])
        face_tensor = torch.tensor(face).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
        embedding = resnet(face_tensor)
        output = classifier(embedding)
        probs = output.detach().cpu().numpy()[0]
        identity = dataset.classes[np.argmax(probs)]
        confidence = probs[np.argmax(probs)] * 100
        print(f'Identity: {identity}, Confidence: {confidence:.2f}%')
cap.release()

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
import torch
import numpy as np

# Inference on video
cap = cv2.VideoCapture('/content/drive/MyDrive/Face_Recog/test_video/adarsh_video.mp4')
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert to RGB for MTCNN processing
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    cv2_imshow(frame_rgb)

    # Detect faces and landmarks
    boxes, _, landmarks = mtcnn.detect(frame_rgb, landmarks=True)

    if boxes is not None:
        x, y, w, h = boxes[0]
        face = frame_rgb[int(y):int(h), int(x):int(w)]  # Extract face from frame
        face = cv2.resize(face, (160, 160))  # Resize face to desired dimensions
        face = align_face(face, landmarks[0])  # Align face based on landmarks

        # Convert to tensor and normalize
        face_tensor = torch.tensor(face).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0

        # Get embedding and classification result
        embedding = resnet(face_tensor)
        output = classifier(embedding)
        probs = output.detach().cpu().numpy()[0]
        identity = dataset.classes[np.argmax(probs)]
        confidence = probs[np.argmax(probs)] * 100

        # Print identity and confidence
        print(f'Identity: {identity}, Confidence: {confidence:.2f}%')

        # Show the face image in a window
        face_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)  # Convert back to BGR for OpenCV
        # cv2_imshow(face_bgr)

    # Break the loop when the user presses the 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
from google.colab.patches import cv2_imshow
import torch
import numpy as np

# Inference on video
cap = cv2.VideoCapture('/content/drive/MyDrive/Face_Recog/test_video/abhinav_video.mp4')
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert to RGB for MTCNN processing
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    cv2_imshow(frame_rgb)

    # Detect faces and landmarks
    boxes, _, landmarks = mtcnn.detect(frame_rgb, landmarks=True)

    if boxes is not None:
        x, y, w, h = boxes[0]
        face = frame_rgb[int(y):int(h), int(x):int(w)]  # Extract face from frame
        face = cv2.resize(face, (160, 160))  # Resize face to desired dimensions
        face = align_face(face, landmarks[0])  # Align face based on landmarks

        # Convert to tensor and normalize
        face_tensor = torch.tensor(face).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0

        # Get embedding and classification result
        embedding = resnet(face_tensor)
        output = classifier(embedding)
        probs = output.detach().cpu().numpy()[0]
        identity = dataset.classes[np.argmax(probs)]
        confidence = probs[np.argmax(probs)] * 100

        # Print identity and confidence
        print(f'Identity: {identity}, Confidence: {confidence:.2f}%')

        # Show the face image in a window
        face_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)  # Convert back to BGR for OpenCV
        # cv2_imshow(face_bgr)

    # Break the loop when the user presses the 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
torch.save({
    'resnet_state_dict': resnet.state_dict(),
    'classifier_state_dict': classifier.state_dict()
}, '/content/drive/MyDrive/Face_Recog/march/model.pth')

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import random

# Load the trained model
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
classifier = FaceClassifier(input_dim=512).to(device)
model_path = '/content/model.pth'  # Adjust to your model file path
checkpoint = torch.load(model_path, map_location=device)
resnet.load_state_dict(checkpoint['resnet_state_dict'])  # If FaceNet was fine-tuned
classifier.load_state_dict(checkpoint['classifier_state_dict'])
resnet.eval()
classifier.eval()

# Dataset class with train/validation split
class FaceDataset(Dataset):
    def __init__(self, root_dir, split='train', train_ratio=0.8):
        self.root_dir = root_dir
        self.classes = ['abhinav', 'andy', 'adarsh', 'adam', 'unknown_subjects']
        self.data = []

        # Load all data
        for idx, cls in enumerate(self.classes):
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.exists(cls_dir):
                print(f"Warning: Directory {cls_dir} does not exist!")
                continue
            for img_name in os.listdir(cls_dir):
                self.data.append((os.path.join(cls_dir, img_name), idx))

        # Shuffle and split
        random.shuffle(self.data)
        split_idx = int(len(self.data) * train_ratio)
        if split == 'train':
            self.data = self.data[:split_idx]
        elif split == 'val':
            self.data = self.data[split_idx:]

        print(f"{split.capitalize()} dataset initialized with {len(self.data)} images across {len(self.classes)} classes: {self.classes}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Failed to load image {img_path}")
            return None, None  # Return None for invalid images
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        boxes, _, landmarks = mtcnn.detect(img, landmarks=True)
        if boxes is None or len(boxes) == 0:
            print(f"No face detected or invalid crop in {img_path}")
            # Return a dummy tensor and a special label to indicate invalid data
            return torch.zeros(3, 160, 160), -1  # Dummy tensor, invalid label
        if boxes is not None and len(boxes) > 0:
            x, y, w, h = boxes[0]  # MTCNN returns [x, y, width, height]
            x, y, w, h = int(x), int(y), int(w), int(h)

            # Compute bottom-right corner
            x2, y2 = x + w, y + h

            # Ensure coordinates are within image bounds
            h_img, w_img = img.shape[:2]
            x = max(0, x)
            y = max(0, y)
            x2 = min(w_img, x2)
            y2 = min(h_img, y2)

            # Check if the region is valid
            if x2 > x and y2 > y:
                face = img[y:y2, x:x2]
                try:
                    face = cv2.resize(face, (160, 160))
                    # Add alignment if needed (assuming align_face is defined)
                    face = torch.tensor(face).permute(2, 0, 1).float() / 255.0
                    return face, label
                except Exception as e:
                    print(f"Error resizing {img_path}: {e}")
                    return None, None  # Return None for invalid images
            else:
                print(f"Invalid crop region for {img_path}: x={x}, y={y}, x2={x2}, y2={y2}")
                return None, None  # Return None for invalid images
        else:
            print(f"No face detected in {img_path}")
            return None, None  # Return None for invalid images

train_dataset = FaceDataset('/content/drive/MyDrive/Face_Recog/march/enrolled_subjects', split='train', train_ratio=0.8)
val_dataset = FaceDataset('/content/drive/MyDrive/Face_Recog/march/enrolled_subjects', split='val', train_ratio=0.8)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Evaluation function
def evaluate_model(loader):
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            images, labels = batch
            # Filter out None values from the batch
            valid_indices = [i for i, img in enumerate(images) if img is not None]
            if not valid_indices:  # Skip batch if all images are None
                continue
            images = images[valid_indices]  # Keep only valid images
            labels = labels[valid_indices]  # Keep only valid labels

            images, labels = images.to(device), labels.to(device)
            embeddings = resnet(images)
            outputs = classifier(embeddings)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Compute metrics
    accuracy = accuracy_score(all_labels, all_preds)
    # Ensure labels are present for precision, recall, f1 calculation
    present_labels = list(set(all_labels))
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=present_labels
    )
    # Ensure all labels are included in confusion matrix
    all_classes = list(range(len(train_dataset.classes)))  # All possible classes
    conf_matrix = confusion_matrix(all_labels, all_preds, labels=all_classes)

    return accuracy, precision, recall, f1, conf_matrix

# Run evaluation
accuracy, precision, recall, f1, conf_matrix = evaluate_model(val_loader)

# Display results
print(f"\nValidation Accuracy: {accuracy * 100:.2f}%")
print("\nPer-class Metrics:")
for i, cls in enumerate(train_dataset.classes):
    # Check if class i is in present_labels
    if i in train_dataset.classes:
        print(f"{cls}:")
        print(f"  Precision: {precision[train_dataset.classes.index(i)]:.4f}")
        print(f"  Recall: {recall[train_dataset.classes.index(i)]:.4f}")
        print(f"  F1-Score: {f1[train_dataset.classes.index(i)]:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix)